<a href="https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage2_memory_agent/Stage2_Memory_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage2_memory_agent/Stage2_Memory_Agent.ipynb)

> **Click the badge above to open this notebook in Google Colab.**
> Or go directly: https://colab.research.google.com/github/appars/codepilot-colab/blob/main/stage2_memory_agent/Stage2_Memory_Agent.ipynb

# 🧠 Stage 2 — Memory Agent
**CodePilot AI Studio | Module 4 | Agentic AI in Software Engineering**

**LLM:** Groq · model `openai/gpt-oss-20b` · key `GROQ_API_KEY` (Colab Secrets)

---

## What You Will Learn
- Why an LLM **forgets** everything — and why that is a problem
- What **short-term memory** means for an AI agent
- How `ConversationBufferMemory` stores the full chat history
- How `PromptTemplate` injects memory into every LLM call automatically
- How `ConversationChain` wires LLM + memory + prompt together

## The Problem We Are Solving
```
WITHOUT memory:                      WITH memory:
Turn 1: 'What is IndexError?'        Turn 1: 'What is IndexError?'
Agent : 'Index out of range...'      Agent : 'Index out of range...'
Turn 2: 'Show me an example'         Turn 2: 'Show me an example'
Agent : 'Example of WHAT?' WRONG!    Agent : 'Here is IndexError...' CORRECT!
```

> **Analogy:** A doctor who reads your full medical history before every appointment.

⏱ **Expected time: 20–25 minutes** (including setup)


## Step 1 — Add Your Groq API Key (One Time Setup)

### Option A — Colab Secrets (Recommended!)
Store your key ONCE in Colab Secrets and it works in ALL notebooks automatically:
1. Click the **🔑 key icon** in the left sidebar
2. Click **'Add new secret'**
3. Name: `GROQ_API_KEY`  (must be exactly this name)
4. Value: paste your key
5. Toggle **'Notebook access'** to ON
6. Come back and run the cell below

### Option B — Type it when asked (hidden input)
If Secrets is not set up, the cell below asks for the key in a hidden input box.
The key is kept only in this running session — it is **not** saved in the notebook.

> 🔐 **Never** paste your key into a code cell, a screenshot, a chat message or GitHub.
> If a key is ever exposed, delete it in the Groq console and create a new one.
>
> Get your free key at **https://console.groq.com** → API Keys → Create API Key


In [ ]:
# ── GROQ API KEY SETUP ───────────────────────────────────────
# 1st choice: Colab Secrets.  2nd choice: hidden input (getpass).
# The key is never printed - not even partly - and never stored in the notebook.

import os
from getpass import getpass

GROQ_API_KEY = None

# ── METHOD 1: Colab Secrets (store once, works in all notebooks)
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    print("Groq API key loaded from Colab Secrets.")
except Exception:
    print("GROQ_API_KEY not found in Colab Secrets (or notebook access is off).")

# ── METHOD 2: Hidden input (nothing is shown on screen or saved in the notebook)
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Paste your Groq API key and press Enter: ").strip()

if not GROQ_API_KEY:
    raise ValueError("No Groq API key provided. Add GROQ_API_KEY in Colab Secrets "
                     "(key icon on the left) or paste it when asked, then re-run this cell.")

# LangChain's ChatGroq reads the key from this environment variable
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("Key is set for this session (hidden). Ready to proceed!")

## Step 2 — Setup

In [ ]:
# Install ONLY what this notebook imports.
# (langchain-community is not needed here; it would upgrade 'requests' and trigger a
#  harmless-but-alarming pip conflict message with Colab's own google-colab package.)
print("Installing packages...")
!pip install -q langchain-groq langchain-core langchain-classic
print("Setup complete!")

## Step 3 — Understand Memory Before Using It

### How ConversationBufferMemory works
```
Every time chain.predict() is called:

 New message ──────────────────────────────────────┐
                                                   ↓
 Memory loads history ──► PromptTemplate ──► LLM ──► Response
  fills {chat_history}     fills {input}              │
                                                       ↓
                                            Memory saves Q+A
```
The key point: the agent re-reads ALL history before every single reply.
That is how it 'remembers' — it does not actually store memories like a human.
It just re-reads the conversation transcript every time!

### The PromptTemplate
```
template = """
You are CodePilot...
Conversation so far:
{chat_history}    ← filled automatically by memory
Student: {input}  ← filled with new message
CodePilot:"""
```

> ℹ️ **Why we hide deprecation warnings:** `ConversationBufferMemory` and `ConversationChain`
> are *legacy* LangChain classes (kept in the `langchain-classic` package). They still work and
> are the clearest way to see memory in action, so the code cells silence the
> `LangChainDeprecationWarning` messages. Newer LangChain/LangGraph code does the same thing with message
> history / checkpointers — the idea (re-inject the history every turn) is identical.


In [ ]:
# ── CONCEPT DEMO: Look inside memory ─────────────────────────
# This cell shows what memory stores - NO LLM call here.

from langchain_classic.memory import ConversationBufferMemory
import warnings
from langchain_core._api import LangChainDeprecationWarning
# These are legacy (but working) LangChain classes - see the note in Step 3.
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

# memory_key="chat_history" MUST match {chat_history} in the prompt template.
# return_messages=False -> history is injected as a readable text transcript
#   (right for a plain-text PromptTemplate).
# human_prefix / ai_prefix -> the names used in that transcript.
demo_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=False,
    human_prefix="Student",
    ai_prefix="CodePilot",
)

# Simulate two turns of conversation by adding messages manually
demo_memory.chat_memory.add_user_message("What is IndexError?")
demo_memory.chat_memory.add_ai_message("IndexError: list index out of range.")
demo_memory.chat_memory.add_user_message("Show me a code example.")
demo_memory.chat_memory.add_ai_message("my_list=[1,2,3]; print(my_list[5]) -> IndexError")

# 1) What is STORED: a list of message objects
print("What memory stores after 2 turns:")
print("-" * 65)
for i, msg in enumerate(demo_memory.chat_memory.messages):
    sender = "Student  " if msg.type == "human" else "CodePilot"
    print(f"[{i+1}] {sender}: {msg.content}")
print(f"\nTotal messages: {len(demo_memory.chat_memory.messages)}")

# 2) What is INJECTED into {chat_history} before the next reply
print("\nWhat gets inserted into {chat_history} in the next prompt:")
print("-" * 65)
print(demo_memory.load_memory_variables({})["chat_history"])

print("\nKEY INSIGHT: The agent re-reads the ENTIRE history before every reply.")
print("That is the whole secret of short-term memory!")

## Step 4 — The Full Memory Agent
Now let us build the real agent with live LLM calls.
Watch how Turn 2 and Turn 3 **reference earlier turns** — proof memory works!

In [ ]:
# ============================================================
# CodePilot AI Studio - Stage 2: Memory Agent
# ============================================================
# Concept  : Short-term memory using ConversationBufferMemory
# New here : ConversationBufferMemory, PromptTemplate, ConversationChain
# LLM      : Groq - openai/gpt-oss-20b
# ============================================================

from langchain_groq import ChatGroq
from langchain_classic.memory import ConversationBufferMemory  # stores chat history
from langchain_classic.chains import ConversationChain         # wires everything together
from langchain_core.prompts import PromptTemplate              # fills in history + message
import warnings
from langchain_core._api import LangChainDeprecationWarning
# These are legacy (but working) LangChain classes - see the note in Step 3.
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

# ── STEP 1: Connect to the LLM ────────────────────────────────
# temperature=0.7 -> good balance for conversational responses
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.7)

# ── STEP 2: Create the memory object ──────────────────────────
# Starts empty. Grows as the conversation continues.
memory = ConversationBufferMemory(
    memory_key="chat_history",   # must match {chat_history} in the template
    return_messages=False,       # inject history as a readable transcript
    human_prefix="Student",
    ai_prefix="CodePilot",
)

# ── STEP 3: Define the prompt template ────────────────────────
# {chat_history} -> auto-filled from memory before every call
# {input}        -> the student's current message
prompt_template = PromptTemplate(
    input_variables=["chat_history", "input"],
    template="""You are CodePilot, a helpful Python coding assistant.
Use the conversation so far to understand follow-up questions. Be concise.

Conversation so far:
{chat_history}

Student: {input}
CodePilot:""",
)

# ── STEP 4: Create the ConversationChain ──────────────────────
# ConversationChain wires together:
#   llm    -> the brain (openai/gpt-oss-20b on Groq)
#   memory -> the notepad (ConversationBufferMemory)
#   prompt -> the template (fills in history + new message)
# After every predict() call, it automatically saves the Q+A to memory.
chain = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt_template,
    verbose=False,   # change to True to see the full prompt each time!
)

# ── STEP 5: Run a 3-turn conversation ─────────────────────────
print("=" * 60)
print("CodePilot AI Studio - Stage 2: Memory Agent")
print("=" * 60)
print()

# Turn 1: first question - memory is empty, the agent answers fresh
q1 = "What is an IndexError in Python?"
print(f"Student (Turn 1): {q1}")
r1 = chain.predict(input=q1)
print(f"CodePilot: {r1}\n")

# Turn 2: 'that error' only makes sense with Turn 1 in memory!
q2 = "Can you give me a code example of that error?"
print(f"Student (Turn 2): {q2}")
r2 = chain.predict(input=q2)   # memory now has Turn 1
print(f"CodePilot: {r2}\n")

# Turn 3: 'the bug you showed me' needs Turns 1 AND 2!
q3 = "How do I fix the bug you just showed me?"
print(f"Student (Turn 3): {q3}")
r3 = chain.predict(input=q3)   # memory has Turns 1 + 2
print(f"CodePilot: {r3}\n")

# Show what is in memory after the conversation
print("=" * 60)
print("Memory contents after 3 turns:")
print("=" * 60)
for i, msg in enumerate(memory.chat_memory.messages):
    sender = "Student  " if msg.type == "human" else "CodePilot"
    print(f"[{i+1}] {sender}: {msg.content[:70]}...")
print(f"\nTotal messages in memory: {len(memory.chat_memory.messages)}")
print()
print("SHORT-TERM MEMORY: lives only while this notebook session runs.")
print("Restart the runtime = memory gone. Surviving a restart needs PERSISTENT")
print("memory - saving the history to storage such as a database.")

## Step 5 — Verify

In [ ]:
# ── VERIFY: did the 3-turn conversation run? ──────────────────
try:
    count = len(memory.chat_memory.messages)
    if count >= 6:
        print("VERIFICATION PASSED")
        print(f"  Turns completed    : {count // 2}")
        print(f"  Messages in memory : {count}")
        print("Ready for the experiments!")
    else:
        print(f"Only {count} messages found. Run Step 4 first.")
except NameError:
    print("Run Step 4 first (the 'memory' object does not exist yet).")

## Step 6 — Experiments

In [ ]:
# ── EXPERIMENT 1: Prove memory works ──────────────────────────
# Ask the agent what the very first question was.
# It should remember because it re-reads all history!

result = chain.predict(input="What was the very first question I asked you?")
print("Test: Can the agent remember Turn 1?")
print(f"CodePilot: {result}")
print()
print("It should mention IndexError - that was Turn 1!")

In [ ]:
# ── EXPERIMENT 2: Turn on verbose mode ────────────────────────
# Shows you the FULL prompt sent to the LLM (openai/gpt-oss-20b).
# This chain has its OWN new, empty memory - watch it fill up.
# This is the most educational experiment in Stage 2!

verbose_chain = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory(memory_key="chat_history", return_messages=False,
                                    human_prefix="Student", ai_prefix="CodePilot"),
    prompt=prompt_template,
    verbose=True,   # <- LOOK at the full prompt printed below!
)
print("=== Turn 1 (verbose mode ON) - chat_history is empty ===")
verbose_chain.predict(input="What is a for loop?")
print()
print("=== Turn 2 - notice chat_history now contains Turn 1! ===")
verbose_chain.predict(input="Show me an example of that.")

In [ ]:
# ── EXPERIMENT 3: Clear memory and test ───────────────────────
# What happens when we wipe memory mid-conversation?

print(f"Before clearing: {len(memory.chat_memory.messages)} messages")
memory.clear()
print(f"After clearing : {len(memory.chat_memory.messages)} messages")
print()

result = chain.predict(input="How do I fix the bug you showed me?")
print("Agent response (after memory clear):")
print(result)
print()
print("Notice: the agent no longer knows which bug you mean - it should ask you,")
print("or give only a generic answer. (LLM wording varies from run to run.)")
print("This PROVES that memory was responsible for remembering before.")

### Experiment 4 (optional) — Watch the prompt grow
`ConversationBufferMemory` keeps **every** message — it never forgets on its own.
So the prompt gets longer every turn. The real limits are the model's **context window**,
**cost** and **latency** — that is why real agents need *context management*.


In [ ]:
# ── EXPERIMENT 4 (optional): the prompt grows every turn ──────
# Uses 3 more API calls.
questions = ["What is a list in Python?", "How do I add an item to it?", "And how do I remove one?"]
for q in questions:
    chain.predict(input=q)
    history = memory.load_memory_variables({})["chat_history"]
    print(f"Messages: {len(memory.chat_memory.messages):2d}  |  "
          f"history injected into the prompt: {len(history):5d} characters")
print("\nThe history only grows. ConversationBufferMemory never drops messages by itself.")

## Summary — Stage 2 Complete!

| Concept | What it means |
|---------|---------------|
| **Stateless LLM** | Forgets after every single call |
| **Short-term memory** | Remembers within one running session — lost on restart |
| **`ConversationBufferMemory`** | Stores all messages; injects them as `{chat_history}` |
| **`PromptTemplate`** | Fills in `{chat_history}` and `{input}` |
| **`ConversationChain`** | Glue connecting LLM + memory + prompt |
| **`chain.predict()`** | Send message → get reply → save to memory |

> Short-term memory ≠ persistent memory. To remember across restarts, an agent must save
> its history to storage (for example a SQLite database — as RootCause Copilot does for incidents).

---
### Problem with Stage 2
One generic prompt handles everything — debug, explain, review — all get the same treatment.

**Stage 3 fixes this with specialised skill prompts and intent detection.**

➡️ Open `stage3_tool_agent/Stage3_Tool_Agent.ipynb`
